In [1]:
!pip install -q -U transformers peft trl datasets accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 116.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 842.4/842.4 kB 66.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 46.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 16.9 MB/s eta 0:00:00


In [10]:
import torch

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4"
)

model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-1.5B-Instruct",
    quantization_config=bnb_config,
    device_map="auto",
)

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct")

model.eval()

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 1536)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear4bit(in_features=1536, out_features=1536, bias=True)
          (k_proj): Linear4bit(in_features=1536, out_features=256, bias=True)
          (v_proj): Linear4bit(in_features=1536, out_features=256, bias=True)
          (o_proj): Linear4bit(in_features=1536, out_features=1536, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear4bit(in_features=1536, out_features=8960, bias=False)
          (up_proj): Linear4bit(in_features=1536, out_features=8960, bias=False)
          (down_proj): Linear4bit(in_features=8960, out_features=1536, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((1

In [16]:
questions = [
    "Produto ótimo, chegou rápido!",
    "Não gostei, veio quebrado",
    "Mais ou menos, esperava mais pelo preço."
]

def ask(_model, tok, review) -> str:
  messages = [
      {"role": "sytem", "content": "Você classifica sentimento de avaliações de produtos"},
      {"role": "user", "content": f'Classifique o sentimento de avaliação à seguir: "{review}"'}
  ]

  inputs = tok.apply_chat_template(
      messages,
      add_generation_prompt=True,
      return_tensors="pt",
      return_dict=True
  )

  with torch.no_grad():
    output = _model.generate(
        **inputs,
        max_new_tokens=60,
        do_sample=False,
        pad_token_id=tok.eos_token_id,
    )

  answers_ids = output[0][inputs["input_ids"].shape[-1]:]

  return tok.decode(answers_ids, skip_special_tokens=True)

baseline = {q: ask(model, tokenizer, q) for q in questions}

In [17]:
baseline

{'Produto ótimo, chegou rápido!': 'O sentimento da avaliação é positivo. A frase indica que o produto foi bem-sucedido e que ele chegou em tempo hábil.',
 'Não gostei, veio quebrado': 'O sentimento da avaliação é negativo. A frase indica que a pessoa não gostou do produto ou serviço e que ele estava danificado.',
 'Mais ou menos, esperava mais pelo preço.': 'O sentimento da avaliação é negativo. A pessoa espera ter obtido algo melhor do que foi recebido.'}

In [19]:
from google.colab import files

uploaded = files.upload()

Saving reviews.parquet to reviews.parquet


In [20]:
import pandas as pd

df = pd.read_parquet("reviews.parquet")
df.head()

,review,sentiment
0,"Produto ótimo, chegou rápido!",positivo
1,"Superou minhas expectativas, recomendo demais.",positivo
2,Chegou antes do prazo e bem embalado.,positivo
3,Qualidade excelente pelo preço que paguei.,positivo
4,"Já é a terceira vez que compro, nunca decepciona.",positivo


In [21]:
from datasets import load_dataset

dataset = load_dataset("parquet", data_files="reviews.parquet", split="train")

Generating train split: 0 examples [00:00, ? examples/s]

In [22]:
def toChatFormat(line):
  return {
      "messages": [
          {"role":"system", "content": "Você classifica sentimento de avaliações de produtos"},
          {"role":"user", "content": line["review"]},
          {"role":"assistant", "content": f'{{"sentiment": "{line["sentiment"]}"}}'},
      ]
  }

dataset_chat = dataset.map(toChatFormat)

Map:   0%|          | 0/60 [00:00<?, ? examples/s]

In [23]:
dataset_chat[0]

{'review': 'Produto ótimo, chegou rápido!',
 'sentiment': 'positivo',
 'messages': [{'role': 'system',
   'content': 'Você classifica sentimento de avaliações de produtos'},
  {'role': 'user', 'content': 'Produto ótimo, chegou rápido!'},
  {'role': 'assistant', 'content': '{"sentiment": "positivo"}'}]}

In [27]:
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

trainable params: 2,179,072 || all params: 1,545,893,376 || trainable%: 0.1410


In [29]:
from trl import SFTConfig, SFTTrainer

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset_chat,
    args=SFTConfig(
        output_dir="./lora-output",
        num_train_epochs=3,
        per_device_train_batch_size=4,
        learning_rate=2e-4
    )
)

trainer.train()
trainer.save_model("./lora-adapter")

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
10,3.487037
20,2.479157
30,1.830237
40,1.401450


In [30]:
import shutil
from google.colab import files

shutil.make_archive("lora-adapter", "zip", "./lora-adapter")
files.download("lora-adapter.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [33]:
model.gradient_checkpointing_disable()
model.config.use_cache = True

model.eval()

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(151936, 1536)
        (layers): ModuleList(
          (0-27): 28 x Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=1536, out_features=1536, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=1536, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=1536, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): Line

In [35]:
def compare(question: str):
  with model.disable_adapter():
    answer_base = ask(model, tokenizer, question)

  answer_tuned = ask(model, tokenizer, question)

  print(f"Avaliação: {question}")
  print(f"Base: {answer_base}")
  print(f"Tuned: {answer_tuned}")

for question in questions:
  compare(question)

/usr/local/lib/python3.12/dist-packages/transformers/generation/utils.py:2613: UserWarning: You are calling .generate() with the `input_ids` being on a device type different than your model's device. `input_ids` is on cpu, whereas the model is on cuda. You may experience unexpected behaviors or slower generation. Please make sure that you have put `input_ids` to the correct device by calling for example input_ids = input_ids.to('cuda') before running `.generate()`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Avaliação: Produto ótimo, chegou rápido!
Base: O sentimento da avaliação é positivo. A frase indica que o produto foi bem-sucedido e que ele chegou rapidamente ao cliente.
Tuned: {"sentiment": "positivo"}
Avaliação: Não gostei, veio quebrado
Base: O sentimento da avaliação é negativo. A frase indica que a pessoa não gostou do produto ou serviço e que ele estava danificado.
Tuned: {"sentiment": "negativo"}
Avaliação: Mais ou menos, esperava mais pelo preço.
Base: O sentimento da avaliação é negativo. A pessoa espera ter obtido algo melhor do que foi recebido.
Tuned: {"sentiment": "neutro"}


In [37]:
def evaluate(_test_cases: list[dict], apply_adapter: bool) -> float:
  corrects = 0

  for test_case in _test_cases:
    if (apply_adapter):
      answer = ask(model, tokenizer, test_case["input"])
    else:
      with model.disable_adapter():
        answer = ask(model, tokenizer, test_case["input"])

    if (test_case["expect"] in answer):
      corrects+=1

  return corrects / len(_test_cases) * 100

test_cases = [
    {"input":"Chegou no prazo e funcionou perfeitamente.", "expect": "positivo"},
    {"input": "Veio com a tela rachada.", "expect": "negativo"}
]

print(f"Base: {evaluate(test_cases, apply_adapter=False):.1f}%")
print(f"Tuned: {evaluate(test_cases, apply_adapter=True):.1f}%")

/usr/local/lib/python3.12/dist-packages/transformers/generation/utils.py:2613: UserWarning: You are calling .generate() with the `input_ids` being on a device type different than your model's device. `input_ids` is on cpu, whereas the model is on cuda. You may experience unexpected behaviors or slower generation. Please make sure that you have put `input_ids` to the correct device by calling for example input_ids = input_ids.to('cuda') before running `.generate()`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Base: 100.0%
Tuned: 100.0%
